In [1]:
# -*- coding: utf-8 -*-
import os
import gc
import json
import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==============================================================================
# CONFIGURATION
# ==============================================================================
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ STEAD 3C_ test n15275 r100/STEAD data, test n15275 r100.json'

PREPROCESSED_H5 = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_50k_safe.h5'

def main():
    os.makedirs(os.path.dirname(PREPROCESSED_H5), exist_ok=True)
    
    # --------------------------------------------------------------------------
    # FASE 1: FILTERING (Pandas)
    # --------------------------------------------------------------------------
    print("[INFO] FASE 1: Memuat metadata...", flush=True)
    with open(ZHI_GENG_JSON, 'r') as f:
        zhi_geng_traces = set(json.load(f).keys())
        
    # Memuat CSV dengan sangat hemat memori
    df_raw = pd.read_csv(CSV_PATH, usecols=['trace_name', 'trace_category', 'p_arrival_sample'])
    df_unseen = df_raw[(df_raw['trace_category'].isin(['earthquake_local', 'noise'])) & 
                       (~df_raw['trace_name'].isin(zhi_geng_traces))]
    
    # Sampling 1K EQ + 1K Noise
    df_final = pd.concat([
        df_unseen[df_unseen['trace_category'] == 'earthquake_local'].sample(n=1000, random_state=42),
        df_unseen[df_unseen['trace_category'] == 'noise'].sample(n=1000, random_state=42)
    ]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    total_target = len(df_final)
    print(f"[INFO] Total data yang akan diproses: {total_target} sampel.", flush=True)
    
    # Bersihkan memori Pandas
    del df_raw, df_unseen
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 2: SEQUENTIAL EXTRACTION (HDF5)
    # --------------------------------------------------------------------------
    print(f"[INFO] FASE 2: Mengekstrak gelombang ke {PREPROCESSED_H5}...", flush=True)
    
    num_points = 700
    norm_points = 900
    berhasil = 0

    with h5py.File(PREPROCESSED_H5, 'w') as f_out:
        # Menyiapkan kerangka dataset kosong
        dset_1c = f_out.create_dataset("X_1C", shape=(total_target, 700, 1), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_3c = f_out.create_dataset("X_3C", shape=(total_target, 700, 3), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_y  = f_out.create_dataset("Y", shape=(total_target,), dtype=np.int32)
        dset_names = f_out.create_dataset("trace_name", shape=(total_target,), dtype=h5py.string_dtype(encoding='utf-8'))
        
        with h5py.File(HDF5_PATH, 'r') as f_in:
            data_group = f_in['data']
            
            # Loop sekuensial (Satu per satu, paling aman)
            for idx, row in tqdm(df_final.iterrows(), total=total_target, desc="Processing HDF5"):
                trace_id = row['trace_name']
                category = row['trace_category']
                p_arrival = 0 if pd.isna(row['p_arrival_sample']) else row['p_arrival_sample']
                
                if trace_id not in data_group:
                    continue
                    
                # Load wave ke RAM sesaat
                raw_wave = data_group[trace_id][()].astype(np.float32)
                raw_wave -= np.mean(raw_wave, axis=0) # Detrend
                
                if category == 'earthquake_local':
                    start = int(p_arrival)
                    if start < 0 or (start + norm_points) > len(raw_wave): 
                        del raw_wave
                        continue
                    wave = raw_wave[start:start+num_points, :]
                    norm_val = np.max(np.abs(raw_wave[start:start+norm_points, :]), axis=0)
                    label = 1
                else:
                    wave = raw_wave[:num_points, :]
                    norm_val = np.max(np.abs(raw_wave[:norm_points, :]), axis=0)
                    label = 0
                    
                norm_val[norm_val == 0] = 1e-8
                wave /= norm_val
                
                wave_1c = wave[:, 2].reshape(num_points, 1)
                
                # Tulis langsung ke disk
                dset_names[berhasil] = trace_id
                dset_3c[berhasil] = wave
                dset_1c[berhasil] = wave_1c
                dset_y[berhasil] = label
                berhasil += 1
                
                # Bersihkan variabel per iterasi
                del raw_wave, wave, wave_1c
                
                # Pembersihan paksa berkala
                if berhasil > 0 and berhasil % 2500 == 0:
                    gc.collect()
                    
        # Resizing file HDF5 output sesuai jumlah data yang benar-benar sukses terekstrak
        if berhasil < total_target:
            dset_1c.resize((berhasil, 700, 1))
            dset_3c.resize((berhasil, 700, 3))
            dset_y.resize((berhasil,))
            dset_names.resize((berhasil,))

    print("\n=======================================================")
    print(f" [SUKSES] Dataset {berhasil} sampel telah di-freeze secara Sekuensial!")
    print("=======================================================")

if __name__ == "__main__":
    main()

[INFO] FASE 1: Memuat metadata...
[INFO] Total data yang akan diproses: 2000 sampel.
[INFO] FASE 2: Mengekstrak gelombang ke /Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_50k_safe.h5...


Processing HDF5: 100%|██████████| 2000/2000 [00:25<00:00, 79.07it/s]



 [SUKSES] Dataset 2000 sampel telah di-freeze secara Sekuensial!


In [1]:
import pandas as pd

CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'

df = pd.read_csv(CSV_PATH, nrows=5)
print(df.columns)
df.head()


Index(['network_code', 'receiver_code', 'receiver_type', 'receiver_latitude',
       'receiver_longitude', 'receiver_elevation_m', 'p_arrival_sample',
       'p_status', 'p_weight', 'p_travel_sec', 's_arrival_sample', 's_status',
       's_weight', 'source_id', 'source_origin_time',
       'source_origin_uncertainty_sec', 'source_latitude', 'source_longitude',
       'source_error_sec', 'source_gap_deg',
       'source_horizontal_uncertainty_km', 'source_depth_km',
       'source_depth_uncertainty_km', 'source_magnitude',
       'source_magnitude_type', 'source_magnitude_author',
       'source_mechanism_strike_dip_rake', 'source_distance_deg',
       'source_distance_km', 'back_azimuth_deg', 'snr_db', 'coda_end_sample',
       'trace_start_time', 'trace_category', 'trace_name'],
      dtype='object')


,network_code,receiver_code,receiver_type,receiver_latitude,receiver_longitude,receiver_elevation_m,p_arrival_sample,p_status,p_weight,p_travel_sec,...,source_magnitude_author,source_mechanism_strike_dip_rake,source_distance_deg,source_distance_km,back_azimuth_deg,snr_db,coda_end_sample,trace_start_time,trace_category,trace_name
0,AE,113A,HH,32.768299,-113.766701,118.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-15 00:17:30,noise,113A.AE_20180115001730_NO
1,AE,113A,HH,32.768299,-113.766701,118.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-15 00:33:36,noise,113A.AE_20180115003336_NO
2,AE,113A,HH,32.768299,-113.766701,118.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-15 02:01:06,noise,113A.AE_20180115020106_NO
3,AE,113A,HH,32.768299,-113.766701,118.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-15 02:29:06,noise,113A.AE_20180115022906_NO
4,AE,113A,HH,32.768299,-113.766701,118.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-15 03:51:00,noise,113A.AE_20180115035100_NO


In [2]:
# -*- coding: utf-8 -*-
import os
import json
import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm

CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ STEAD 3C_ test n15275 r100/STEAD data, test n15275 r100.json'

OUTPUT_JSON = '/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_MCQUAKE_5000_unseen.json'

NUM_POINTS = 700
NORM_POINTS = 900

def extract_window(raw, start):
    """Ambil window 700 sampel + normalisasi max abs 900 sampel."""
    norm_val = np.max(np.abs(raw[start:start+NORM_POINTS, :]), axis=0)
    norm_val[norm_val == 0] = 1e-8
    raw_norm = raw / norm_val
    return raw_norm[start:start+NUM_POINTS, :]

def main():
    print("[INFO] Load JSON Zhi Geng...")
    with open(ZHI_GENG_JSON, 'r') as f:
        used_by_zhi = set(json.load(f).keys())
    print(f"[INFO] Total trace yang dipakai Zhi Geng: {len(used_by_zhi)}")

    print("[INFO] Load metadata STEAD...")
    df = pd.read_csv(CSV_PATH, usecols=['trace_name', 'trace_category', 'p_arrival_sample'])

    # Filter kategori + exclude Zhi Geng
    df = df[
        (df['trace_category'].isin(['earthquake_local', 'noise'])) &
        (~df['trace_name'].isin(used_by_zhi))
    ].reset_index(drop=True)

    print(f"[INFO] Total STEAD yang belum pernah dipakai Zhi Geng: {len(df)}")

    # Ambil 2500 LE + 2500 NO
    df_noise = df[df['trace_category'] == 'noise'].sample(n=2500, random_state=42)
    df_eq    = df[df['trace_category'] == 'earthquake_local'].sample(n=2500, random_state=42)

    df_final = pd.concat([df_noise, df_eq]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"[INFO] Total target final: {len(df_final)} traces")

    output = {}

    with h5py.File(HDF5_PATH, 'r') as f_in:
        data_group = f_in['data']

        for _, row in tqdm(df_final.iterrows(), total=len(df_final), desc="STEAD→JSON"):
            trace_id = row['trace_name']
            category = row['trace_category']
            p_arrival = 0 if pd.isna(row['p_arrival_sample']) else int(row['p_arrival_sample'])

            if trace_id not in data_group:
                continue

            raw = data_group[trace_id][()].astype(np.float32)
            raw -= np.mean(raw, axis=0)

            # Tentukan start window
            if category == 'earthquake_local':
                start = p_arrival
                label = "se"
            else:
                start = 0
                label = "no"

            # Validasi panjang
            if start < 0 or (start + NUM_POINTS) > len(raw):
                continue

            # Ekstraksi window
            wave = extract_window(raw, start)

            # Ambil komponen Z (index 2)
            Z = wave[:, 2].tolist()

            output[trace_id] = {
                "type": label,
                "Z": Z
            }

    # Simpan JSON
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    print(f"[SUKSES] JSON STEAD disimpan ke: {OUTPUT_JSON}")
    print(f"[TOTAL] {len(output)} sampel berhasil diekspor.")

if __name__ == "__main__":
    main()


[INFO] Load JSON Zhi Geng...
[INFO] Total trace yang dipakai Zhi Geng: 15275
[INFO] Load metadata STEAD...
[INFO] Total STEAD yang belum pernah dipakai Zhi Geng: 1265657
[INFO] Total target final: 5000 traces


STEAD→JSON: 100%|██████████| 5000/5000 [01:00<00:00, 82.91it/s]


[SUKSES] JSON STEAD disimpan ke: /Volumes/Extreme SSD/stream_stead/data_stead/STEAD_MCQUAKE_5000_unseen.json
[TOTAL] 5000 sampel berhasil diekspor.
